# SHAPE — Self-Healing Autonomous Programming Engine
## Notebook 1 of 4 · Introduction & Architecture

**Group 3 | ITAI2376-14562**  
Cherluk Sumdin · Olubukola Kukoyi · Taki Eddine Boubekri · Michael Rivas

---

**Project option:** Option 2 — Automation Agent  
**What this notebook covers:** Purpose, problem statement, hardware architecture, data flow,
package structure, and agent lifecycle. No agent is initialized here — that is Notebook 02.

**One shared package:** All four notebooks import from the same `shape/` package.
There is one `SHAPEAgent`, one `parse_llm_output`, one `InputGuard`, one `FeedbackMemory`.


## 1. Problem Statement

Every popular AI coding assistant (GitHub Copilot, ChatGPT, Claude) requires:
- A live internet connection
- A paid API subscription ($10–$60/month)
- Sending your code to a remote server

This makes them **completely unavailable** in real-world offline environments:
wildfire mapping crews with no cell signal, air-gapped hospital networks,
remote Antarctic research stations, and embedded Pi-based robotics kits.

**SHAPE's answer:** put the assistant on the device. If the Raspberry Pi is already
in the field doing sensor work, run the agent on a companion Pi.
No cloud. No subscription. No data leaving the device.


## 2. Project Option: Option 2 — Automation Agent

| Requirement | SHAPE implementation | File |
|---|---|---|
| Monitor an environment | Pico 2 reads DHT22 + MQ-135 every 2 s | `hardware/pico2/sensors.py` |
| Take autonomous actions | Auto-alert queries when AQI > 600 or temp > 40°C | `hardware/pi5/api_server.py:/sensor` |
| Adapt over time | RL feedback loop adjusts system prompt between sessions | `shape/rewards.py` |
| Use ≥ 2 external tools | 4 tools: code_executor, linter, file_manager, doc_search | `shape/tools/` |
| Handle errors / fallbacks | Linter pre-screens; response cache if Pi 5 offline | `shape/tools/linter.py`, `hardware/zero2w/` |
| Input validation | InputGuard regex screen (pre-LLM) + telemetry range checks | `shape/safety.py`, `shape/sensors.py` |


## 3. Hardware Architecture — Three Tiers

```
┌─────────────────────────────────────────────────────────────────────┐
│  TIER 1 — Raspberry Pi Pico 2 (RP2350, MicroPython)                │
│  hardware/pico2/main.py  sensors.py  keys.py                       │
│  ● DHT22 → temperature, humidity  ● MQ-135 → AQI                  │
│  ● Waveshare 1.3" ST7789 LCD (240×240)                             │
│  ● Sends CSV every 2 s:  SHAPE,{ts},{temp},{humidity},{aqi},{alert} │
└───────────────────────────────┬─────────────────────────────────────┘
                        UART 115200 baud (GPIO 0/1)
┌───────────────────────────────▼─────────────────────────────────────┐
│  TIER 2 — Raspberry Pi Zero 2 W (ARM64 Linux)                      │
│  hardware/zero2w/main_zero2w.py  serial_bridge.py  mic_recorder.py │
│  ● Waveshare HAT: ST7789 LCD + joystick + K1/K2/K3 buttons         │
│  ● ShapeTerminal state machine: HOME→QUERY→THINKING→RESPONSE→RATE  │
│  ● K3 long-press → mic_recorder → WAV → POST /transcribe           │
│  ● ResponseCache: last 10 answers on SD (offline fallback)         │
└───────────────────────────────┬─────────────────────────────────────┘
                        HTTP Wi-Fi  192.168.4.1:8765
┌───────────────────────────────▼─────────────────────────────────────┐
│  TIER 3 — Raspberry Pi 5 (8 GB, ARM64 Linux)                       │
│  hardware/pi5/api_server.py    shape/agent.py                       │
│  ● Ollama + CodeLlama 7B (q4_K_M, ~3.8 GB)                        │
│  ● SHAPEAgent.run() — ReAct loop, 4 tools, RL feedback             │
│  ● 6 HTTP endpoints: /query /feedback /status /reset /sensor /transcribe │
└─────────────────────────────────────────────────────────────────────┘
```


## 4. Numbered Data Flow: Sensor Reading → LCD Output

| Step | Location | Action | Code |
|---|---|---|---|
| 1 | Pico 2 | DHT22 + MQ-135 sampled every 2 s | `sensors.py:SensorHub.measure()` |
| 2 | Pico 2 | CSV line sent over UART | `main.py:_send_csv()` |
| 3 | Zero 2 W | `SerialBridge` parses CSV → `SensorReading` | `serial_bridge.py` |
| 4 | Zero 2 W | If `aqi > 600` or `temp > 40°C`: POST `/sensor` | `serial_bridge.py` → `Pi5Client` |
| 5 | Pi 5 | **Layer 1 (deterministic):** threshold rule fires `cmd='ALERT'` | `api_server.py` |
| 6 | Pi 5 | **Layer 2 (LLM):** `agent.run(auto_query)` generates advisory text | `shape/agent.py` |
| 7 | Pi 5 | Response + RL reward packaged as `{cmd, text, reward}` | `api_server.py` |
| 8 | Zero 2 W | `SerialBridge.push_to_pico(text, reward)` → UART | `serial_bridge.py` |
| 9 | Pico 2 | `_recv_json()` reads command; LCD updated | `main.py:_recv_json()` |
| 10 | Pico 2 LCD | Researcher sees: alert text + AQI colour + reward bar | `main.py:draw_sensor_screen()` |

**Alert reliability:** Step 5 fires before the LLM. The decision to alert is always
rule-based arithmetic — the LLM only generates the advisory text.


## 5. Package Structure — One Codebase, Four Notebooks

```
shape/
├── __init__.py        public API exports
├── agent.py           SHAPEAgent + build_agent() — the only constructor
├── parser.py          parse_llm_output() — Notebook 02
├── memory.py          make_history(), trimmed_messages() — Notebook 02
├── prompts.py         REACT_SYSTEM template — Notebook 02
├── rewards.py         FeedbackMemory — Notebook 04
├── safety.py          InputGuard — Notebook 03
├── sensors.py         SensorReading, parse_csv_line() — Notebook 03
└── tools/
    ├── __init__.py    get_tools(config)
    ├── code_executor.py
    ├── linter.py
    ├── file_manager.py
    └── doc_search.py

tests/
├── test_parser.py    9 tests — all branches
├── test_agent.py     12 tests — safety, memory, rewards
└── test_tools.py     10 tests — all error paths
```

Every notebook does `from shape.xxx import yyy`.
No notebook reimplements any logic.


In [ ]:
# ── Colab / environment setup ─────────────────────────────────────────────────
# Run once per session. Skip if already in SHAPE_Core directory.
import os, sys
from pathlib import Path

if not Path('shape').exists():
    # Colab: clone repo and install dependencies
    import subprocess
    subprocess.run(['git', 'clone',
                    'https://github.com/mrivassnj-svg/SHAPE_Core.git'], check=True)
    os.chdir('SHAPE_Core')
    # Pin langchain-ollama to avoid HTTP 400 tools API error with CodeLlama
    subprocess.run(['pip', 'install', '-r', 'requirements.txt', '--quiet'], check=True)
    subprocess.run(['pip', 'install', 'langchain-ollama==0.1.3',
                    '--quiet', '--force-reinstall'], check=True)

# Always ensure the package root is on sys.path
root = Path('.').resolve()
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

print('Working directory:', os.getcwd())
print('shape/ found:', Path('shape').exists())

In [ ]:
# Verify the package structure — all modules must be importable from shape/

from shape          import __version__, __authors__
from shape.parser   import parse_llm_output
from shape.memory   import make_history, trimmed_messages
from shape.prompts  import REACT_SYSTEM
from shape.rewards  import FeedbackMemory, MIN_RECORDS_FOR_HINT
from shape.safety   import InputGuard, _BLOCKED_PATTERNS, _OUT_OF_SCOPE_PATTERNS
from shape.sensors  import parse_csv_line, SensorReading, AQI_ALERT_THRESHOLD, TEMP_ALERT_THRESHOLD
from shape.tools    import get_tools
from shape.agent    import build_agent, SHAPEAgent

print(f'SHAPE v{__version__}')
print(f'Authors: {", ".join(__authors__)}')
print()
print('Module inventory:')
print(f'  shape.parser   — parse_llm_output()    ← Notebook 02')
print(f'  shape.memory   — make_history()         ← Notebook 02')
print(f'  shape.prompts  — REACT_SYSTEM           ← Notebook 02')
print(f'  shape.rewards  — FeedbackMemory         ← Notebook 04')
print(f'  shape.safety   — InputGuard ({len(_BLOCKED_PATTERNS)} blocked, {len(_OUT_OF_SCOPE_PATTERNS)} out-of-scope patterns)  ← Notebook 03')
print(f'  shape.sensors  — parse_csv_line()       ← Notebook 03')
print(f'  shape.tools    — get_tools()            ← Notebook 03')
print(f'  shape.agent    — build_agent()          ← Notebooks 02–04')

In [ ]:
# Run the test suite — 21 tests covering parser, safety, memory, tools.
# All must pass before proceeding to Notebook 02.

import subprocess, sys
result = subprocess.run(
    [sys.executable, '-m', 'pytest', 'tests/', '-v', '--tb=short', '-q'],
    capture_output=True, text=True
)
print(result.stdout[-3000:] if len(result.stdout) > 3000 else result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr[-500:])
    raise RuntimeError('Tests failed — do not proceed until all pass.')
print('All tests passed. ✓')

In [ ]:
# Demonstrate the Pico 2 UART CSV protocol via shape.sensors

from shape.sensors import parse_csv_line, validate_reading

sample_lines = [
    'SHAPE,14520,22.40,58.1,312,0',   # normal
    'SHAPE,16520,24.10,60.0,648,1',   # AQI alert
    'SHAPE,18520,41.2,55.0,290,0',    # temp alert
    'GARBAGE,bad,data',               # malformed
    'SHAPE,20520,-999,55.0,100,0',    # out-of-range temp
]

print(f'{"Line":<40} {"Parsed?":<8} {"Alert?":<8} {"Warnings"}')
print('-' * 85)
for line in sample_lines:
    r = parse_csv_line(line)
    if r is None:
        print(f'  {line:<38}  None     —        malformed — silently dropped')
    else:
        warnings = validate_reading(r)
        print(f'  {line:<38}  OK       {str(r.is_alert):<8} {warnings or "none"}')

In [ ]:
# Interactive ReAct Visualizer
# Teaching mode = deterministic scenarios (no LLM)
# Live Trace mode = observes real agent.run() output

from pathlib import Path
from IPython.display import display, HTML, IFrame

viz = Path('notebooks/shape_react_visualizer.html')
if not viz.exists():
    viz = Path('shape_react_visualizer.html')

if viz.exists():
    try:
        import google.colab  # noqa
        display(HTML(viz.read_text()))
    except ImportError:
        display(IFrame(src=str(viz), width='100%', height='820px'))
    print('Visualizer loaded.')
    print('  Teaching mode: deterministic — safe for grading/demo')
    print('  Live Trace:    connects to actual agent.run() in Notebook 02')
else:
    print('Open notebooks/shape_react_visualizer.html in a browser.')

## Summary

| Item | Demonstrated |
|---|---|
| Project option | Option 2 — Automation Agent, all 6 requirements mapped |
| Three-tier hardware | Pico 2 → Zero 2 W → Pi 5 architecture diagram |
| Data flow | 10-step numbered sequence from sensor sample to LCD output |
| Alert reliability | Layer 1 (rule-based arithmetic) vs Layer 2 (LLM advisory) |
| Package structure | 9 modules, one import path per notebook |
| Tests | 21 unit tests — all pass before proceeding |
| Sensor protocol | `shape.sensors.parse_csv_line()` live-tested including malformed input |

▶ **Continue to `02_SHAPE_ReAct_Engine.ipynb`**
